← [Previous: Chapter 4: SystemVerilog Elaborated Netlist](https://colab.research.google.com/github/najaeda/naja/blob/main/tutorials/notebooks/04_systemverilog_elaborated_netlist.ipynb) · 🔗 [Back to Table of Contents](https://github.com/najaeda/naja/tree/main/tutorials)

# Chapter 5: Exploring ibex, an Open-Source RISC-V Core

[ibex](https://github.com/lowRISC/ibex) is a production-grade 32-bit RISC-V core written in SystemVerilog, developed by lowRISC.
Loading it demonstrates `najaeda`'s ability to handle real-world, parameterised SV designs: packages, generate blocks, complex port buses, and a multi-stage pipeline hierarchy.

In this chapter we will:
1. Clone ibex at a pinned commit and load it with `najaeda`
2. Inspect the top-level RISC-V bus interface
3. Navigate the pipeline hierarchy below `ibex_top`
4. Collect module statistics across the full design
5. Dump the elaborated netlist as Verilog

---

## Setting Up the Environment

In [ ]:
# Colab / local setup — this cell is skipped in CI (environment is pre-configured)
import os
from pathlib import Path

!pip install najaeda 'fusesoc>=2.4'

# Clone ibex at the pinned commit known to work with najaeda
!git clone https://github.com/lowRISC/ibex.git ibex
!cd ibex && git checkout eede2fbbef007d53cafbd85d937b897751c40a54

# Generate the file list with fusesoc (lint target, setup only — no actual linting)
!cd ibex && fusesoc --cores-root . run \
    --target=lint \
    --setup \
    --build-root build/fusesoc \
    --mapping=lowrisc:prim_generic:all:0.1 \
    lowrisc:ibex:ibex_top

os.environ['IBEX_REPO'] = str(Path('ibex').resolve())

## Loading the Design

We find the fusesoc-generated `.vc` file and convert its relative paths to absolute ones before handing it to `najaeda`.

In [ ]:
import os
from pathlib import Path
from najaeda import netlist


def _normalize_flist(vc_path: Path) -> str:
    """Convert a fusesoc .vc file to an absolute-path flist string."""
    base = vc_path.parent
    lines = []
    for raw in vc_path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('+incdir+'):
            p = line[len('+incdir+'):]
            lines.append(f'+incdir+{(base / p).resolve()}')
        elif line.startswith('+define+'):
            lines.append(line)
        elif line.startswith('-D') and len(line) > 2:
            lines.append('+define+' + line[2:])
        elif line.endswith(('.sv', '.v', '.svh', '.vh')):
            lines.append(str((base / line).resolve()))
    return '\n'.join(lines)


ibex_repo = Path(os.environ.get('IBEX_REPO', 'ibex'))
vc_file = next(ibex_repo.glob('build/fusesoc/**/lint-verilator/*.vc'))
flist_content = _normalize_flist(vc_file)

flist_path = ibex_repo / 'ibex_top_normalized.flist'
flist_path.write_text(flist_content)
print(f"Flist written to: {flist_path}")
print(f"  {len(flist_content.splitlines())} entries")

netlist.reset()
top = netlist.load_system_verilog(
    [],
    config=netlist.SystemVerilogConfig(
        flist=str(flist_path),
        top='ibex_top',
        suppress_warnings=['sign-conversion'],
    ),
)
print(f"\nLoaded top: {top.get_name()} (model: {top.get_model_name()})")
print(f"Ports: {top.count_terms()} | direct children: {top.count_child_instances()}")

---

## Inspecting the Top-Level RISC-V Interface

`ibex_top` exposes three main bus groups: an **instruction bus** (OBI protocol), a **data bus** (OBI protocol), and a set of **interrupt / debug** control signals. The cell below prints each port with its direction and width.

In [ ]:
def port_dir(term):
    direction = term.get_direction()
    if 'INPUT' in str(direction):
        return 'input '
    elif 'OUTPUT' in str(direction):
        return 'output'
    return '  ?'

print(f"{'Direction':<5}  {'Width':>5}  Name")
print('-' * 40)
for term in top.get_terms():
    width = term.get_width() if term.is_bus() else 1
    print(f"  {port_dir(term)}  {width:>5}  {term.get_name()}")

---

## First-Level Hierarchy

`ibex_top` is a thin wrapper. The pipeline logic lives inside `ibex_core`. Let us look at the direct children of `ibex_top` and then open up `ibex_core`.

In [ ]:
print("Children of ibex_top:")
for child in top.get_child_instances():
    if child.is_primitive():
        continue
    nb_children = child.count_child_instances()
    print(f"  {child.get_name():<35} model: {child.get_model_name()} "
          f"({nb_children} sub-instances)")

# Locate ibex_core (it is the child whose model contains 'ibex_core')
ibex_core = next(
    c for c in top.get_child_instances()
    if 'ibex_core' in (c.get_model_name() or '')
)
print(f"\nChildren of {ibex_core.get_name()} ({ibex_core.get_model_name()}):")
for child in ibex_core.get_child_instances():
    if child.is_primitive():
        continue
    print(f"  {child.get_name():<35} model: {child.get_model_name()}")

---

## Navigating the Fetch Path

We descend from `ibex_core` into the instruction-fetch stage and then into the prefetch buffer to inspect its ports and internal nets.

In [ ]:
# Find the IF (instruction-fetch) stage inside ibex_core
if_stage = next(
    c for c in ibex_core.get_child_instances()
    if 'if_stage' in (c.get_name() or '') or 'if_stage' in (c.get_model_name() or '')
)
print(f"IF stage: {if_stage.get_name()} (model: {if_stage.get_model_name()})")
print(f"  ports: {if_stage.count_terms()}, children: {if_stage.count_child_instances()}")

# Locate the prefetch buffer inside the IF stage
prefetch = next(
    (c for c in if_stage.get_child_instances()
     if 'prefetch' in (c.get_model_name() or '')),
    None,
)
if prefetch:
    print(f"\nPrefetch buffer: {prefetch.get_name()} (model: {prefetch.get_model_name()})")
    print(f"  ports:")
    for term in prefetch.get_terms():
        width = term.get_width() if term.is_bus() else 1
        print(f"    {port_dir(term)}  {width:>4}  {term.get_name()}")
else:
    print("(prefetch buffer not found at this level)")

---

## Design-Wide Module Statistics

The visitor API lets us traverse the full instance tree in one pass and collect counts per module type.

In [ ]:
from collections import Counter
from najaeda import instance_visitor

model_counter = Counter()

def count_model(instance):
    model_counter[instance.get_model_name() or '<unnamed>'] += 1

visitor_config = instance_visitor.VisitorConfig(callback=count_model)
instance_visitor.visit(top, visitor_config)

total_instances = sum(model_counter.values())
print(f"Total instances (all levels): {total_instances}")
print(f"Unique module types:          {len(model_counter)}")
print(f"\nTop 15 most-instantiated modules:")
print(f"  {'Module':<45} {'Count':>6}")
print('  ' + '-' * 53)
for model_name, count in model_counter.most_common(15):
    print(f"  {model_name:<45} {count:>6}")

---

## Dumping the Elaborated Netlist

`dump_verilog` serialises the elaborated (parameter-substituted) netlist back to a Verilog file. The output captures the concrete port widths and hierarchy that resulted from elaboration — useful as input to downstream tools or for comparison with the original source.

In [ ]:
out_file = 'ibex_top_elaborated.v'
top.dump_verilog(out_file)

import os
size_kb = os.path.getsize(out_file) / 1024
print(f"Written: {out_file} ({size_kb:.1f} KB)")

# Show the first 40 lines as a preview
with open(out_file) as f:
    preview = [next(f) for _ in range(40)]
print('\n'.join(l.rstrip() for l in preview))
print('...')

---

✅ This concludes **Chapter 5** of the **najaeda** tutorial.

You have seen how to load a real-world, parameterised SystemVerilog core into `najaeda`, navigate its pipeline hierarchy, collect design-wide module statistics using the visitor API, and export the elaborated netlist as Verilog.

➡️ [**Next: Chapter 6 — Gate-Level Analysis: Fanout and Net Statistics**](https://colab.research.google.com/github/najaeda/naja/blob/main/tutorials/notebooks/06_fanout_analysis.ipynb)
